# Lesson 1: Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [2]:
client = OpenAI()

In [3]:
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [4]:
chat_completion.choices[0].message.content

'Hello! How can I assist you today?'

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [ ]:
abot = Agent(prompt)

In [ ]:
result = abot("How much does a toy poodle weigh?")
print(result)

In [ ]:
result = average_dog_weight("Toy Poodle")

In [ ]:
result

In [ ]:
next_prompt = "Observation: {}".format(result)

In [ ]:
abot(next_prompt)

In [ ]:
abot.messages

In [ ]:
abot = Agent(prompt)

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

In [ ]:
abot(next_prompt)

### Add loop 

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [ ]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

In [12]:
print(chat_completion.choices[0].message.content)

Hello! How can I assist you today?


In [13]:
## Lets define our own agent to do this

In [36]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        
        # if the system is defined
        if self.system:
            self.messages.append({"role": "system", "content": system})
    
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        completion = client.chat.completions.create(
            model="gpt-4-0125-preview",
            temperature=0,
            messages = self.messages)
        return completion.choices[0].message.content
        

In [37]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [38]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [39]:
abot = Agent(prompt)

In [40]:
print(abot.execute())

What is the question you'd like to ask?


In [41]:
result = abot("What is the weight of Toy Poddle")

In [42]:
result = average_dog_weight("Toy Poodle")

In [43]:
print(result)

a toy poodles average weight is 7 lbs


In [44]:
next_prompt = "Observation: {}".format(average_dog_weight("Toy Poodle"))

In [ ]:
abot(next_prompt)

In [46]:
print(abot)

In [48]:
abot.messages
print(abot.messages)

[{'role': 'system', 'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'}, {'role': 'user', 'content': 'What is the weight 

In [49]:
# Reinit the agent
llm_response = Agent(prompt)

In [52]:
llm_response("How much does Scottish Terrier weight")

'Action: average_dog_weight: Scottish Terrier'

In [51]:
print(result)

Action: average_dog_weight: Scottish Terrier


In [59]:
# Here basically it decides which function to call
# either call caluclate or the average_dog_weight

result = average_dog_weight("Scottish Terrier")

In [60]:
print(next_prompt)

Scottish Terriers average 20 lbs


In [62]:
# feed the llm as the observation

next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))

In [63]:
llm_response(next_prompt)

'Answer: Scottish Terriers average 20 lbs.'

In [64]:
llm_response.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does Sc

In [65]:
llm = Agent(prompt)

In [66]:
llm("Calculate 2 + 4")

'Action: calculate: 2 + 4'

In [68]:
result = calculate('2+4')

In [69]:
print(result)

6


In [70]:
next_prompt = "Observation: {}".format(calculate("2+4"))

In [71]:
llm(next_prompt)

'Answer: 6'

In [72]:
llm.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'Calculate 2 + 4'

In [73]:
complex_llm = Agent(prompt)

In [74]:
complex_llm("What is the avg weight of Scottish Terrier, Border Collie, Toy Poodle")

'Thought: I need to find the average weight of three dog breeds: Scottish Terrier, Border Collie, and Toy Poodle. I will start by finding the average weight of a Scottish Terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [75]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))

In [76]:
complex_llm(next_prompt)

'Thought: Now that I know the average weight of a Scottish Terrier, I need to find the average weight of a Border Collie next.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [77]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))

In [78]:
complex_llm(next_prompt)

'Thought: With the average weight of Scottish Terriers and Border Collies known, the next step is to find the average weight of a Toy Poodle.\nAction: average_dog_weight: Toy Poodle\nPAUSE'

In [79]:
next_prompt = "Observation: {}".format(average_dog_weight("Toy Poodle"))

In [80]:
complex_llm(next_prompt)

'Thought: Now that I have the average weights of Scottish Terriers (20 lbs), Border Collies (37 lbs), and Toy Poodles (7 lbs), I can calculate the average weight of these three dog breeds.\nAction: calculate: (20 + 37 + 7) / 3\nPAUSE'

In [81]:
next_prompt = "Observation: {}".format(eval("(20+37+7)//3"))

In [82]:
complex_llm(next_prompt)

'Answer: The average weight of a Scottish Terrier, Border Collie, and Toy Poodle is 21 lbs.'

In [83]:
complex_llm.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user',
  'content': 'What is the av

In [ ]:
## Lets build a function to achieve this programitically

In [84]:
action_re = re.compile('^Action: (\w+): (.*)$')

In [88]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    
    while i <= max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        
        print('--- actions --- {}'.format(actions))
        
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception('Unknown action {}: {}'.format(action, action_input))
            
            print('--- running -- {} {}'.format(action, action_input))
            observation = known_actions[action](action_input)
            print('Observation:', observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [89]:
question = """What is the avg weight of Scottish Terrier, Border Collie, Toy Poodle"""

In [90]:
query(question)

Thought: I need to find the average weight of three dog breeds: Scottish Terrier, Border Collie, and Toy Poodle. I will start by finding the average weight of a Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE
--- actions --- [<re.Match object; span=(0, 44), match='Action: average_dog_weight: Scottish Terrier'>]
--- running -- average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: Now that I know the average weight of a Scottish Terrier, I need to find the average weight of a Border Collie next.
Action: average_dog_weight: Border Collie
PAUSE
--- actions --- [<re.Match object; span=(0, 41), match='Action: average_dog_weight: Border Collie'>]
--- running -- average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: With the average weight of Scottish Terriers and Border Collies known, the next step is to find the average weight of a Toy Poodle.
Action: average_dog_weight: Toy Poodle
PAUSE
--- a